# Representation-level dissociation test (ACML revision)

Answers reviewers **R1.2 / R2.1 / R3.1**: the submitted study varied only last-layer heads, so
"not a representation problem" was read as unsupported. Here the **backbone itself** is fine-tuned
end-to-end under ERM / GroupDRO / group-balanced objectives, and the full head x calibration
comparison is re-run on each resulting representation.

**The verdict is falsifiable.** Two levers are compared head to head:

* *representation lever* -- the best worst-group coverage reachable under **marginal** calibration, maximising over representations and heads
* *calibration lever* -- the worst worst-group coverage reached under **Mondrian**, minimising over representations and heads

If the worst calibration cell still beats the best representation cell (CI-separated), the title
stands. If a robust representation closes the marginal gap on its own, the title must narrow.
Both outcomes are reported.

**Runtime (L4).** Waterbirds ~45 min total. CelebA ~2.5-3 h. Every fine-tune checkpoints to Drive
after each epoch, so a session that hits its limit **resumes** rather than restarting -- just
re-run the cell. Fine-tuned features are cached to Drive too, so a second session is nearly free.

Run Waterbirds -> **STOP and review** -> then CelebA.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
REPO_SOURCE   = "git"          # "git" | "drive"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE  = ""

# --- fine-tune budget (L4 defaults) -------------------------------------------------
FT_OBJECTIVES = ("erm", "groupdro", "reweight")
FT_SEEDS      = (0, 1, 2, 3, 4)  # 5 seeds: R1.4 asked for more, and the equivalence test needs
                                 # them -- at 3 seeds a 0.013 spread still bootstraps to hi=0.031,
                                 # just missing the 0.030 margin. Waterbirds is ~2.2 min/run.

# Per-objective recipes. One shared schedule is NOT the fairer control: on ERM's Adam recipe the
# GroupDRO arm came out *less* robust than ERM (eval wg acc 0.619 vs 0.748), i.e. a failed
# manipulation that would make the null coverage result uninformative. Sagawa et al. (2020) show
# GroupDRO on Waterbirds needs SGD with strong L2 or it overfits the minority group.
#   erm/reweight: weight_decay 0.0 reproduces the backbone the submitted paper already used
#   (features.resnet50_erm_features = Adam(params, lr) with no decay). Adding L2 inside Adam
#   measurably lowered worst-group accuracy and broke comparability with the paper's own tables.
FT_HP = {
    "erm":      dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "reweight": dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "groupdro": dict(optimizer="sgd",  lr=1e-3, weight_decay=1e-2, groupdro_eta=0.05,
                     epochs_override=20),
}
WB_EPOCHS     = 10             # Waterbirds: 4,795 imgs -> ~3-5 min/run on L4
CELEBA_EPOCHS = 5              # CelebA: 162,770 imgs -> ~25-35 min/run on L4
CELEBA_MAX_TRAIN = None        # set e.g. 50000 to cut CelebA cost ~3x (documented subsample)
BATCH_SIZE    = 128
NUM_WORKERS   = 8              # decode is the bottleneck on CelebA, not the GPU
LR            = 1e-3
CACHE_DTYPE   = "float16"      # on-disk feature cache: float16 halves Drive usage (14 GB -> 7 GB
                               # for CelebA) at ~1e-3 relative precision on L2-normalised features.
                               # Set "float32" if Drive space is not a constraint.
N_SPLITS      = 10
HEADS         = ("erm", "dfr", "groupdro_ll")
SCORES        = ("APS", "RAPS", "THR")

## 1. Drive + repo

`init_drive()` does not just call `drive.mount` -- it writes, reads back and deletes a probe file,
because the mount point can exist while every write to it fails. It force-remounts and retries on
failure, then reports free space.

**If you hit "A Google Drive error has occurred":**

| cause | check | fix |
|---|---|---|
| Drive full | free space printed by this cell and cell 2 | `CACHE_DTYPE="float16"`, lower `CELEBA_MAX_TRAIN`, or free space |
| stale / half-dead mount | probe write fails, retries print | Runtime > *Disconnect and delete runtime*, re-run |
| authorisation expired | mount prompt reappears or throws | re-accept the Drive permission prompt |
| too much small-file I/O | error appears mid-run, not at mount | already mitigated: checkpoints are local and writes are atomic |

The `sh()` helper here raises on a non-zero exit code and prints stderr. The earlier version
printed only stdout, so a failed `ln -s` looked identical to a successful one -- which is how a
Drive problem turns into a confusing result several cells later.

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    """Run a shell command and SHOW failures. A silent helper is how a Drive error becomes an
    unexplained wrong result three cells later, so stderr and the exit code are never swallowed."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed (rc={r.returncode}): {cmd}")
    return r.returncode == 0

def init_drive(mount="/content/drive", retries=3):
    """Mount Google Drive and PROVE it works before anything depends on it.

    'A Google Drive error has occurred' usually means the FUSE mount is present but not actually
    serving I/O. Checking os.path.exists() is not enough -- the directory can exist while every
    write fails. So we write, read back, and delete a probe file, and force a remount if that
    round-trip fails.
    """
    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            drive.mount(mount, force_remount=attempt > 1)
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            probe = os.path.join(DRIVE_CACHE, ".mount_probe")
            token = str(time.time())
            with open(probe, "w") as fh: fh.write(token)
            with open(probe) as fh: got = fh.read()
            os.remove(probe)
            if got != token: raise IOError("probe read-back mismatch")
            free_gb = os.statvfs(mount).f_bavail * os.statvfs(mount).f_frsize / 1e9
            print(f"Drive OK (attempt {attempt}) -> {DRIVE_CACHE} | ~{free_gb:.1f} GB free")
            return True
        except Exception as e:
            print(f"[drive] attempt {attempt}/{retries} failed: {e}")
            time.sleep(5 * attempt)
    raise RuntimeError(
        "Google Drive would not mount after retries. Fix before continuing:\n"
        "  1. Runtime > Disconnect and delete runtime, then re-run this cell\n"
        "  2. Check Drive storage is not full (this notebook writes ~1-2 GB of caches)\n"
        "  3. Re-authorise the Drive permission prompt when it appears\n"
        "  4. If it still fails, set REPO_SOURCE/caches to local /content paths and accept "
        "that nothing survives the session.")

init_drive()

REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

import torch
print("repo:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (fix runtime!)")

## 2. Drive-backed caches + space check

All four caches are symlinked to Drive and each link is **verified** by writing a probe file
through it. `cache_finetune` and `study` matter most: the first makes a re-run nearly free, the
second means the results CSV survives the session. *(The original grid notebook Drive-backed only
the feature caches, which is why `grid_records.csv` did not persist.)*

**Checkpoints deliberately stay on local disk.** They only enable resume *within* a partially
finished run; the durable artifact is the feature cache, which is on Drive. Routing ~100 MB/epoch
of checkpoint traffic through Drive buys little and is a common trigger for
*"A Google Drive error has occurred"*.

This cell also projects how much Drive space the run needs and tells you what to change if there
is not enough -- CelebA feature caches are 14 GB in float32 across nine runs, which alone exceeds
a free 15 GB Drive, hence the `float16` default.

In [ ]:
os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_finetune", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"
    os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}")                      # removes the LINK, never the Drive contents
    sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe"             # prove the link resolves onto Drive
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe"), f"results/{c} does not resolve to Drive"
    os.remove(probe)
print("all four caches symlinked to Drive and verified")

# Checkpoints stay on LOCAL disk. They only enable resume *within* a partially finished run, while
# the durable artifact -- the extracted feature cache -- already lives on Drive, so a lost session
# costs at most one run. Putting 100 MB/epoch of checkpoint traffic on Drive instead buys little and
# is a common trigger for "A Google Drive error has occurred".
CKPT_DIR = "/content/ft_ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)

free_gb = os.statvfs("/content/drive").f_bavail * os.statvfs("/content/drive").f_frsize / 1e9
need_wb = 0.10 * len(FT_OBJECTIVES) * len(FT_SEEDS)
need_cel = (1.58 if CACHE_DTYPE == "float32" else 0.79) * len(FT_OBJECTIVES) * len(FT_SEEDS)
print(f"\nDrive free: {free_gb:.1f} GB")
print(f"projected feature cache: Waterbirds ~{need_wb:.1f} GB + CelebA ~{need_cel:.1f} GB "
      f"= ~{need_wb + need_cel:.1f} GB  (CACHE_DTYPE={CACHE_DTYPE})")
if free_gb < need_wb + need_cel:
    print("\n*** NOT ENOUGH DRIVE SPACE. Options, cheapest first:\n"
          "      - set CACHE_DTYPE = \"float16\"  (halves it; features are L2-normalised so the\n"
          "        precision loss is far below existing noise)\n"
          "      - set CELEBA_MAX_TRAIN = 50000   (smaller train split, documented subsample)\n"
          "      - drop FT_SEEDS to (0, 1)        (weakens the CIs -- last resort)\n"
          "      - free space in Drive, or point DRIVE_CACHE at a roomier account")
sh("ls -la results/")

## 3. Datasets

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("WATERBIRDS_ROOT =", os.environ["WATERBIRDS_ROOT"], "| CelebA OK =", CELEBA_OK)

## 4. Sanity gate -- validate the analysis machinery before spending GPU time

In [ ]:
rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_representation"],
                    capture_output=True, text=True)
print(rc.stdout[-3000:])
assert rc.returncode == 0, "validator FAILED -- do not burn GPU time until this passes"

## 4b. Stale caches from the first run (optional cleanup)

The first Waterbirds run used `weight_decay=1e-4` inside Adam. That is now known to be wrong: the
submitted paper's backbone (`features.resnet50_erm_features`) uses `Adam(params, lr)` with **no**
decay, and adding L2 inside Adam measurably lowered worst-group accuracy -- the new ERM arm scored
0.276/0.748/0.381 where the paper's tables report 0.510/0.812/0.656. Those caches are therefore not
comparable with the paper and **should** be recomputed rather than reused.

`cache_key` now records `weight_decay`, `optimizer` and (for GroupDRO) `groupdro_eta`, so the stale
files simply miss and everything rebuilds on the corrected recipe. Nothing to migrate.

This cell only deletes the orphaned files to reclaim Drive space. It is safe to skip -- they are a
few hundred MB on Waterbirds -- but do run it before CelebA if space is tight.

In [ ]:
import glob, re
STALE = re.compile(r"^ft-\w+?_.+?_\d+ep_lr[\d.e-]+_bs\d+(_mtall|_mt\d+)_s\d+\.")  # no _wd.. segment
freed = 0
for p in sorted(glob.glob("results/cache_finetune/*")):
    b = os.path.basename(p)
    if os.path.isfile(p) and STALE.match(b):
        freed += os.path.getsize(p); os.remove(p); print("  removed stale:", b)
print(f"\nreclaimed {freed/1e9:.2f} GB of pre-fix caches")
print("remaining:", len(glob.glob("results/cache_finetune/*.npz")), "feature file(s)")

## 5. Waterbirds -- fine-tune 3 objectives x 5 seeds

~2.2 min per run on an L4; 3 objectives x 5 seeds is ~35 min, less whatever the cache already
holds. Re-running this cell after an interruption resumes from the last completed epoch.

**The manipulation check at the top of the report is the gate.** It must say PASS: at least one
robust objective has to beat the ERM representation on *eval* worst-group accuracy. If it says
FAIL, the null coverage result below it is uninformative about representations and cannot be used
in the rebuttal -- fix the recipe in `FT_HP` before spending the CelebA budget.

Judge the arms on that eval table, never on `train-wg`: a 10-epoch fine-tune memorises the training
split, which is why ERM reaches `train-wg` 0.91 while being the non-robust arm.

Also watch `q=[...]` on the GroupDRO lines -- if the weights stay near `[0.25,0.25,0.25,0.25]` the
robust objective is not reweighting at all; raise `groupdro_eta` in `FT_HP`.

In [ ]:
from study_robust_train.representation import build_repr_griddata

def cfg_for(dataset, epochs, max_train=None):
    base = {"finetune": {"device": "cuda", "epochs": epochs,
                         "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "amp": True,
                         "max_train": max_train, "cache_dir": "results/cache_finetune",
                         "ckpt_dir": CKPT_DIR, "cache_dtype": CACHE_DTYPE}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
    return base

def build_all(dataset, epochs, max_train=None):
    cfg, data, failed = cfg_for(dataset, epochs, max_train), {}, []
    for obj in FT_OBJECTIVES:
        hp = dict(FT_HP.get(obj, {}))
        ep = hp.pop("epochs_override", epochs)     # robust arms may need a longer schedule
        for s in FT_SEEDS:
            t = time.time()
            try:
                data[(dataset, obj, s)] = build_repr_griddata(dataset, obj, cfg, ft_seed=s,
                                                              epochs=ep, **hp)
                print(f"[built] {dataset}/{obj}/s{s}  ({(time.time()-t)/60:.1f} min)", flush=True)
            except Exception as e:
                failed.append((dataset, obj, s, repr(e))); print(f"[FAIL] {dataset}/{obj}/s{s}: {e}")
    return data, failed

wb_data, wb_failed = build_all("waterbirds", WB_EPOCHS)
print("\nbuilt:", len(wb_data), "| failed:", wb_failed)

## 6. Run the comparison on Waterbirds

In [ ]:
from study_robust_train.representation import run_representation_experiment, write_representation_md
from study_robust_train.grid import write_csv
from IPython.display import Markdown, display

wb_out = run_representation_experiment(wb_data, heads=HEADS, scores=SCORES, n_splits=N_SPLITS)
write_csv(wb_out["records"], "results/study/representation_records.csv")
display(Markdown(write_representation_md(wb_out, "REPRESENTATION.md")))

## 7. STOP -- review before spending the CelebA budget

Read the two-lever verdict above. Either outcome is publishable, but they lead to different
revisions, so decide the framing now rather than after another 3 GPU-hours:

* **CALIBRATION LEVER DOMINATES** -> the title survives; CelebA becomes replication.
* **representation lever competitive** -> the title must narrow, and CelebA then matters *more*,
  because the claim becomes dataset-specific and needs a second dataset to characterise.

Also sanity-check that the fine-tune actually did something: `train-wg` in the logs above should
rise for `groupdro` / `reweight` relative to `erm`. If it did not, the robust objective never
changed the representation and the comparison is vacuous -- raise `groupdro_eta` or switch to
`optimizer="sgd"` before spending the CelebA budget.

## 8. CelebA (heavier -- ~2.5-3 h on L4; resumable)

In [ ]:
assert CELEBA_OK, "CelebA unavailable -- set CELEBA_SOURCE / upload kaggle.json"
cel_data, cel_failed = build_all("celeba", CELEBA_EPOCHS, CELEBA_MAX_TRAIN)
print("\nbuilt:", len(cel_data), "| failed:", cel_failed)

## 9. Combined report (Waterbirds + CelebA)

In [ ]:
all_data = {**wb_data, **cel_data}
out = run_representation_experiment(all_data, heads=HEADS, scores=SCORES, n_splits=N_SPLITS)
write_csv(out["records"], "results/study/representation_records.csv")
display(Markdown(write_representation_md(out, "REPRESENTATION.md")))
print("\nPersisted to Drive:", f"{DRIVE_CACHE}/study/representation_records.csv")

## 10. Pull the older CSVs down as well

The bucket-B reanalysis (TOST, cluster bootstrap, correlation CIs) needs the *original* grid
records. This lists what actually survived on Drive.

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/", check=False)
for f in ("representation_records.csv", "grid_records.csv", "calibration_ablation.csv",
          "predicted_group_mondrian.csv"):
    p = f"{DRIVE_CACHE}/study/{f}"
    print(("FOUND   " if os.path.exists(p) else "MISSING ") + p)